In [1]:
from loader import load_dataset
from precompute_s2_labels import load_s2_labels, lookup_s2_id
import torch
from transformers import AutoImageProcessor, ConvNextForImageClassification, PretrainedConfig

In [ ]:
#Constants
DATASET_PATH = "C:/GeoDataset/dataset_sharded_TEST"
S2_LABEL_PATH = "./L6_TEST.csv"
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LEARNING_RATE = 5e-5

In [ ]:
#Labels
'''
lookup function that maps a panoid to a numerical class index based on its corresponding S2 cell ID (location).

tables precalculated and stored in memory for fast lookup
'''

s2_table = load_s2_labels(S2_LABEL_PATH) #Table: [panoid, s2_id]
unique_s2_ids = list(set(s2_table.values()))

class_index_table = {} #Table: [s2_id, index]
for i, s2_id in enumerate(unique_s2_ids):
    class_index_table[s2_id] = i

num_classes = len(set(class_index_table))

def lookup_class_index(panoid: str): #panoid -> index
    s2_id = lookup_s2_id(s2_table, panoid)
    return class_index_table.get(s2_id)


In [4]:
dataset = load_dataset(
    dataset_path=DATASET_PATH, 
    lookup_class_label_from_panoid_method=lookup_class_index,
    shuffle=False
)

loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=0)

#Test
for imgs, labels in loader:
    print("Batch images:", imgs.shape)
    print("Batch labels:", labels)
    break

Batch images: torch.Size([32, 3, 768, 768])
Batch labels: tensor([ 0,  0,  5,  5,  3,  3,  3,  0,  5,  4,  2,  5,  1,  2,  4,  6,  7,  8,
         9,  8,  9,  9,  9,  8,  7,  8,  7, 10, 10, 10, 10, 11])


In [7]:
PRETRAINED_MODEL_ID   = "facebook/convnext-tiny-224"

processor = AutoImageProcessor.from_pretrained(PRETRAINED_MODEL_ID)
model = ConvNextForImageClassification.from_pretrained(
    PRETRAINED_MODEL_ID,
    num_labels=num_classes,
    ignore_mismatched_sizes=True
).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

Some weights of ConvNextForImageClassification were not initialized from the model checkpoint at facebook/convnext-tiny-224 and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([503, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([503]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from tqdm import tqdm
import torch.nn.functional as F

model.train()

EPOCHS = 2

for epoch in range(EPOCHS):  
    total_loss = 0.0

    for imgs, labels in tqdm(loader, desc=f"Epoch {epoch+1}"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        outputs = model(imgs)
        loss = F.cross_entropy(outputs.logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1} | Average Loss: {avg_loss:.4f}")


In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        outputs = model(imgs)
        preds = torch.argmax(outputs.logits, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total
print(f"Evaluation accuracy: {accuracy:.2f}%")
print(f"min acc: {1/num_classes}")
